# 05. Export data for the county explorer

The Streamlit app (`app/explorer.py`) runs on Streamlit Community Cloud straight
from GitHub, where `medicare.duckdb` does not exist (it is gitignored). So this
notebook writes the small slice of data the app needs to two Parquet files in
`app/data/`, which are committed with the app:

* **`counties.parquet`**: one row per county-year, with spending (total and by
  type of service), the matching variables, and their z-scores
* **`national.parquet`**: national spending per beneficiary by year, for context

Parquet is a compressed column format. DuckDB reads and writes it natively.

Needs `merged` (notebook 02) and `raw` (notebook 01).

In [ ]:
import os

import duckdb

con = duckdb.connect("../medicare.duckdb", read_only=True)
os.makedirs("../app/data", exist_ok=True)

## 1. How "similar counties" are defined

Matching uses only the explanatory variables from the regression, **never spending**,
so the comparison answers: *among counties like this one, does it spend more or less?*

| Variable | Column |
|---|---|
| median household income | `median_income` |
| poverty rate | `poverty_rate` |
| average beneficiary age | `avg_age` |
| % dual-eligible | `dual_pct` |
| % in Medicare Advantage | `ma_rate` |
| rural-urban code (1-9) | `rucc` |

Each variable becomes a z-score **within its year** (subtract that year's mean,
divide by that year's standard deviation), so a 2024 county is only compared with
2024 counties and dollar values are not distorted by inflation. The app then picks
the counties with the smallest Euclidean distance across the six z-scores.

In SQL, `AVG(x) OVER w` with `WINDOW w AS (PARTITION BY year, matchable)`
computes the mean for each year without collapsing rows, like `egen mean = mean(x), by(year)` in Stata.
Counties missing any matching variable (`matchable = false`) get no z-scores
and are not included in anyone's comparison group.

In [ ]:
con.sql("""
COPY (
    WITH base AS (
        SELECT m.year, m.fips, m.state,
               SUBSTR(m.county_name, 4) || ', ' || m.state AS name,   -- 'GA-Miller' -> 'Miller, GA'
               m.benes, m.spend_pc_std,
               m.median_income, m.poverty_rate, m.avg_age,
               m.dual_pct * 100 AS dual_pct,          -- shares -> percent
               m.ma_rate  * 100 AS ma_rate,
               m.rucc, m.rucc_desc, m.metro_rural,
               TRY_CAST(r.IP_MDCR_STDZD_PYMT_PC AS DOUBLE) AS ip,
           TRY_CAST(r.OP_MDCR_STDZD_PYMT_PC AS DOUBLE) AS op,
           TRY_CAST(r.SNF_MDCR_STDZD_PYMT_PC AS DOUBLE) AS snf,
           TRY_CAST(r.HH_MDCR_STDZD_PYMT_PC AS DOUBLE) AS hh,
           TRY_CAST(r.HOSPC_MDCR_STDZD_PYMT_PC AS DOUBLE) AS hospice,
           TRY_CAST(r.EM_MDCR_STDZD_PYMT_PC AS DOUBLE) AS physician,
           TRY_CAST(r.PRCDRS_MDCR_STDZD_PYMT_PC AS DOUBLE) AS procedures,
           TRY_CAST(r.TESTS_MDCR_STDZD_PYMT_PC AS DOUBLE) AS tests,
           TRY_CAST(r.IMGNG_MDCR_STDZD_PYMT_PC AS DOUBLE) AS imaging
        FROM merged m
        LEFT JOIN raw r
          ON r.BENE_GEO_LVL = 'County' AND r.BENE_AGE_LVL = 'All'
         AND r.YEAR = m.year AND r.BENE_GEO_CD = m.fips
    ),
    flagged AS (
        SELECT *,
               spend_pc_std - (COALESCE(ip, 0) + COALESCE(op, 0) + COALESCE(snf, 0) + COALESCE(hh, 0) + COALESCE(hospice, 0) + COALESCE(physician, 0) + COALESCE(procedures, 0) + COALESCE(tests, 0) + COALESCE(imaging, 0)) AS other,
               (median_income IS NOT NULL
           AND poverty_rate IS NOT NULL
           AND avg_age IS NOT NULL
           AND dual_pct IS NOT NULL
           AND ma_rate IS NOT NULL
           AND rucc IS NOT NULL) AS matchable
        FROM base
    )
    SELECT *,
       CASE WHEN matchable THEN (median_income - AVG(median_income) OVER w) / STDDEV_SAMP(median_income) OVER w END AS z_income,
       CASE WHEN matchable THEN (poverty_rate - AVG(poverty_rate) OVER w) / STDDEV_SAMP(poverty_rate) OVER w END AS z_poverty,
       CASE WHEN matchable THEN (avg_age - AVG(avg_age) OVER w) / STDDEV_SAMP(avg_age) OVER w END AS z_age,
       CASE WHEN matchable THEN (dual_pct - AVG(dual_pct) OVER w) / STDDEV_SAMP(dual_pct) OVER w END AS z_dual,
       CASE WHEN matchable THEN (ma_rate - AVG(ma_rate) OVER w) / STDDEV_SAMP(ma_rate) OVER w END AS z_ma,
       CASE WHEN matchable THEN (rucc - AVG(rucc) OVER w) / STDDEV_SAMP(rucc) OVER w END AS z_rucc
    FROM flagged
    WINDOW w AS (PARTITION BY year, matchable)
    ORDER BY year, fips
) TO '../app/data/counties.parquet' (FORMAT parquet)
""")

In [ ]:
con.sql("""
COPY (
    SELECT YEAR AS year,
       TRY_CAST(TOT_MDCR_STDZD_PYMT_PC AS DOUBLE) AS spend_pc_std,
       TRY_CAST(IP_MDCR_STDZD_PYMT_PC AS DOUBLE) AS ip,
       TRY_CAST(OP_MDCR_STDZD_PYMT_PC AS DOUBLE) AS op,
       TRY_CAST(SNF_MDCR_STDZD_PYMT_PC AS DOUBLE) AS snf,
       TRY_CAST(HH_MDCR_STDZD_PYMT_PC AS DOUBLE) AS hh,
       TRY_CAST(HOSPC_MDCR_STDZD_PYMT_PC AS DOUBLE) AS hospice,
       TRY_CAST(EM_MDCR_STDZD_PYMT_PC AS DOUBLE) AS physician,
       TRY_CAST(PRCDRS_MDCR_STDZD_PYMT_PC AS DOUBLE) AS procedures,
       TRY_CAST(TESTS_MDCR_STDZD_PYMT_PC AS DOUBLE) AS tests,
       TRY_CAST(IMGNG_MDCR_STDZD_PYMT_PC AS DOUBLE) AS imaging
    FROM raw
    WHERE BENE_GEO_LVL = 'National' AND BENE_AGE_LVL = 'All'
    ORDER BY year
) TO '../app/data/national.parquet' (FORMAT parquet)
""")

## 2. Check the export

DuckDB can query a Parquet file directly, as if it were a table.

In [ ]:
for f in ["counties", "national"]:
    path = f"../app/data/{f}.parquet"
    print(f"{path}: {os.path.getsize(path) / 1e6:.2f} MB")

con.sql("""
SELECT year, COUNT(*) AS counties, COUNT(*) FILTER (WHERE matchable) AS matchable,
       ROUND(AVG(z_income), 3) AS mean_z_income, ROUND(STDDEV_SAMP(z_income), 3) AS sd_z_income
FROM '../app/data/counties.parquet'
GROUP BY year ORDER BY year
""")

Each year's z-scores should have mean 0 and standard deviation 1 among matchable counties.

The service categories should add up to no more than the total, so `other` should never be negative:

In [ ]:
con.sql("""
SELECT COUNT(*) FILTER (WHERE other < -1) AS negative_other, ROUND(MIN(other)) AS min_other,
       ROUND(MEDIAN(other / spend_pc_std), 3) AS median_other_share
FROM '../app/data/counties.parquet'
""")

In [ ]:
con.close()